# Estimate Mode Choice

Estimate mode choice models of TNC vs transit/walk

In [39]:
import numpy as np

import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.reset_option('display.float_format')

import biogeme.biogeme as bio
import biogeme.database as biodb
from biogeme import models
from biogeme.expressions import Beta, Variable

In [40]:
# read the data
df = pd.read_csv('out/mode_choice_estimation_file.csv')
df.head()

,hh_id,person_id,person_num,day_id,day_num,joint_trip_id,joint_trip_num,depart_date,depart_hour,depart_minute,depart_seconds,arrive_date,arrive_hour,arrive_minute,arrive_second,distance_meters,distance_miles,duration_minutes,dwell_mins,flag_speed,flag_distance,flag_duration,o_tract_2020,d_tract_2020,hh_member_1,hh_member_2,hh_member_3,hh_member_4,hh_member_5,hh_member_6,hh_member_7,hh_member_8,hh_member_9,hh_member_10,o_purpose,o_purpose_category,d_purpose,d_purpose_category,n_legs,leg_num,first_leg,last_leg,linked_trip_id,linked_trip_num,linked_trip_mode,outbound,joint_status,linked_trip_weight,tour_id,tour_num,linked_trip_mode_labeled,mode,mode2,o_district,d_district,hh_id_tour,person_id_tour,person_num_tour,day_id_tour,day_num_tour,distance_meters_tour,distance_miles_tour,duration_minutes_tour,o_tract_2020_tour,d_tract_2020_tour,num_travelers,num_hh_travelers,joint_status_tour,tour_num_tour,joint_tour_id,tour_start_date,tour_start_hour,tour_start_minute,tour_start_second,tour_end_date,tour_end_hour,tour_end_minute,tour_end_second,tour_category,tour_mode,tour_purpose,partial_status,stops_outbound,stops_inbound,tour_weight,day_of_week,time_period,origin_id,destination_id,ff_car_time_minutes,car_ivt,tnc_wait,tnc_time,tnc_fare,transit_fare,walk_time,transit_time,transit_or_walk_time,walk_faster_than_transit,transit_or_walk_fare,tnc_time_minus_transit_walk,tnc_cost_minus_transit_walk,transit_avail,walk_avail,income_detailed,income_broad,num_vehicles,num_people,num_workers,num_adults,num_kids,hh_weight,income_labeled,tract_2020,hh_share_inc_under_100k,hh_share_inc_over_100k
0,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,11,20,0.0,2024-05-21,11,40,0.0,1304,0.810270,20,5.0,0,0,0,17031320101,17031081500,1,0,0,0,0,0,0,0,0,0,1,1,33,10,4,1,1,0,2400012401010101,1,15,1,1,1853.792592,24000124010101,1,Walk,walk,walk,Downtown,Downtown,24000124,2400012401,1,240001240101,1,3246,2.016976,175,1.703132e+10,1.703132e+10,1,1,1,1,NaN,2024-05-21,11,20,0.0,2024-05-21,14,15,0.0,2,15,10,0,0,2,1237.449397,Tuesday,midday,17031320101,17031081500,3.738333,4.235906,5,9.235905,3.841450,2.5,16.205401,22.0,16.205401,True,0.0,-6.969495,3.841450,1,1,12,5,2,2,1,2,0,794.392,"$150,000 or more",17031320101,0.341000,0.659000
1,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,11,45,0.0,2024-05-21,12,13,0.0,544,0.338027,28,82.0,0,0,0,17031081500,17031081403,1,0,0,0,0,0,0,0,0,0,33,10,150,12,4,2,0,0,2400012401010102,2,15,0,1,1853.792592,24000124010101,1,Walk,walk,walk,Downtown,Downtown,24000124,2400012401,1,240001240101,1,3246,2.016976,175,1.703132e+10,1.703132e+10,1,1,1,1,NaN,2024-05-21,11,20,0.0,2024-05-21,14,15,0.0,2,15,10,0,0,2,1237.449397,Tuesday,midday,17031081500,17031081403,1.660000,1.880946,5,6.880946,2.742938,2.5,6.760535,7.0,6.760535,True,0.0,0.120411,2.742938,1,1,12,5,2,2,1,2,0,794.392,"$150,000 or more",17031081500,0.321678,0.678322
2,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,13,35,0.0,2024-05-21,13,50,0.0,884,0.549293,15,9.0,0,0,0,17031081403,17031320101,1,0,0,0,0,0,0,0,0,0,150,12,33,10,4,3,0,0,2400012401010103,3,15,0,1,1853.792592,24000124010101,1,Walk,walk,walk,Downtown,Downtown,24000124,2400012401,1,240001240101,1,3246,2.016976,175,1.703132e+10,1.703132e+10,1,1,1,1,NaN,2024-05-21,11,20,0.0,2024-05-21,14,15,0.0,2,15,10,0,0,2,1237.449397,Tuesday,midday,17031081403,17031320101,3.421667,3.877090,5,8.877090,3.507735,2.5,10.985870,23.0,10.985870,True,0.0,-2.108779,3.507735,1,1,12,5,2,2,1,2,0,794.392,"$150,000 or more",17031081403,0.460539,0.539461
3,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,13,59,0.0,2024-05-21,14,15,0.0,514,0.319386,16,NaN,0,0,0,17031320101,17031320101,1,0,0,0,0,0,0,0,0,0,33,10,1,1,4,4,0,1,2400012401010104,4,15,0,1,1853.792592,24000124010101,1,Walk,walk,walk,Downtown,Downtown,24000124,2400012401,1,240001240101,1,3246,2.016976,175,1.703132e+10,1.703132e+10,1,1,1,1,NaN,2024-05-21,11,20,0.0,2024-05-21,14,15,0.0,2,15,10,0,0,2,1237.449397,Tuesday,midday,17031320101,17031320101,2.201667,2.494709,5,7.494708,

In [41]:
# which columns have NaNs, and how many
df.isna().sum()


hh_id                             0
person_id                         0
person_num                        0
day_id                            0
day_num                           0
joint_trip_id                     0
joint_trip_num                 4495
depart_date                       0
depart_hour                       0
depart_minute                     0
depart_seconds                    0
arrive_date                       0
arrive_hour                       0
arrive_minute                     0
arrive_second                     0
distance_meters                   0
distance_miles                    0
duration_minutes                  0
dwell_mins                     1484
flag_speed                        0
flag_distance                     0
flag_duration                     0
o_tract_2020                      0
d_tract_2020                      0
hh_member_1                       0
hh_member_2                       0
hh_member_3                       0
hh_member_4                 

In [42]:
# drop columns with missing values that I dont' need
df = df.drop(columns=['joint_trip_num', 'dwell_mins', 'joint_tour_id', 'o_tract_2020_tour', 'd_tract_2020_tour']).copy()


In [43]:
# calculate normalized weights
df['normalized_weights'] = df['linked_trip_weight'] / df['linked_trip_weight'].sum() * len(df)

In [44]:
# add a flag for trips made by people in HHs with <$100k, $100k+ and missing annual income
# income_broad: 
# 1	Under $30,000
# 2	$30,000-$59,999
# 3	$60,000-$99,999
# 4	$100,000-$149,999
# 5	$150,000 or more
# 999	Prefer not to answer

df['inc_under_100k'] = np.where(df['income_broad']<=3, 1, 0)
df['inc_over_100k']  = np.where((df['income_broad']==4) | (df['income_broad']==5), 1, 0)
df['inc_missing']    = np.where((df['income_broad']==999), 1, 0)

In [45]:
# Biogeme needs a NUMERIC choice column: tnc=1, transit=2, walk=3 and only numeric values in its database format. 
df['CHOICE'] = df['mode'].map({'tnc': 1, 'transit': 2, 'walk': 3})
df['BINARY_CHOICE'] = df['mode'].map({'tnc': 1, 'transit': 2, 'walk': 2})

df_numeric = df.select_dtypes(include='number').copy()

db = biodb.Database('mode_choice', df_numeric)

# Initial Specification

In [46]:
# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST      = Beta('B_COST',      0, None, None, 0)   # generic, shared across modes

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')
CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST * tnc_fare
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST * transit_fare
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, logprob)
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST']['Value']
print("\nValue of Time: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		4
Sample size:			5769
Excluded data:			0
Null log likelihood:		-5835.602
Final log likelihood:		-2506.911
Likelihood ratio test (null):		6657.382
Rho square (null):			0.57
Rho bar square (null):			0.57
Akaike Information Criterion:	5021.822
Bayesian Information Criterion:	5048.463

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  2.068422      0.123294    16.776309           0.0
ASC_WALK     2.982368      0.155110    19.227413           0.0
B_COST      -0.337436      0.031309   -10.777624           0.0
B_TIME      -0.091755      0.006821   -13.452450           0.0

Value of Time: 16.32


# Try weighted estimation

Normally I would use an unweighted estimation.  Here I try a weighted estimation since I think it will affect primarily the ASCs.  What we see below is that it is that the time and cost coefficients are smaller in magnitude, but the ASCs aren't much different.  I think we're better off sticking with the unweighted estimation, which is the norm. 

In [47]:
# Normally I would use an unweighted estimation, but here I care about the ASCs, so I will try weighting it.

# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST      = Beta('B_COST',      0, None, None, 0)   # generic, shared across modes

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')
CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST * tnc_fare
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST * transit_fare
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, {'loglike' : logprob, 'weight' : Variable('normalized_weights')})
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST']['Value']
print("\nValue of Time: " + str(round(vot, 2)))

Results for model mnl_mode_choice
Nbr of parameters:		4
Sample size:			5769
Excluded data:			0
Null log likelihood:		-5835.602
Final log likelihood:		-3097.36
Likelihood ratio test (null):		5476.485
Rho square (null):			0.469
Rho bar square (null):			0.469
Akaike Information Criterion:	6202.719
Bayesian Information Criterion:	6229.36

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  2.304514      0.113027    20.389087           0.0
ASC_WALK     2.911569      0.131201    22.191721           0.0
B_COST      -0.217848      0.024222    -8.993879           0.0
B_TIME      -0.066024      0.006303   -10.475627           0.0

Value of Time: 18.18


# Segment cost coefficient by income

This is backwards of what I would expect--a lower value of time for higher income households.  The fares I'm using are are estimated from 2019 TNC data.  Maybe that's the problem. 

In [48]:
# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers
B_COST_MISSING = Beta('B_COST_MISSING', 0, None, None, 0)   # cost coefficient for travelers with income missing or not reported

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')

inc_under_100k = Variable('inc_under_100k')
inc_over_100k  = Variable('inc_over_100k')
inc_missing    = Variable('inc_missing')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * inc_under_100k + B_COST_HI * tnc_fare * inc_over_100k + B_COST_MISSING * tnc_fare * inc_missing
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * inc_under_100k + B_COST_HI * transit_fare * inc_over_100k + B_COST_MISSING * transit_fare * inc_missing
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, logprob)
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_MISSING']['Value']
print("Value of Time for HH with missing income: " + str(round(vot, 2)))


Results for model mnl_mode_choice
Nbr of parameters:		6
Sample size:			5769
Excluded data:			0
Null log likelihood:		-5835.602
Final log likelihood:		-2488.158
Likelihood ratio test (null):		6694.889
Rho square (null):			0.574
Rho bar square (null):			0.573
Akaike Information Criterion:	4988.315
Bayesian Information Criterion:	5028.277

                   Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT     2.028888      0.121321    16.723302  0.000000e+00
ASC_WALK        2.878925      0.153388    18.768851  0.000000e+00
B_COST_HI      -0.436795      0.037027   -11.796582  0.000000e+00
B_COST_LOW     -0.309908      0.031631    -9.797693  0.000000e+00
B_COST_MISSING -0.446166      0.074595    -5.981189  2.215150e-09
B_TIME         -0.090946      0.006799   -13.376160  0.000000e+00

Value of Time for HH <$100k: 17.61
Value of Time for HH $100k+: 12.49
Value of Time for HH with missing income: 12.23


# Consider zonal incomes instead of HH level incomes

In the TNC data, we won't actually observe the HH level incomes due to privacy restrictions.  Instead try segmenting VOT based on the income distribution in the Census tract of the trip's origin. 

Again, we observe it is kind of backwards, which is strange. Maybe I should not have dropped the short walk trips? 

In [49]:
# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT = Beta('ASC_TRANSIT', 0, None, None, 0)
ASC_WALK    = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time     = Variable('tnc_time')
transit_time = Variable('transit_time')
walk_time    = Variable('walk_time')
tnc_fare     = Variable('tnc_fare')
transit_fare = Variable('transit_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k  = Variable('hh_share_inc_over_100k')

CHOICE       = Variable('CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit = ASC_TRANSIT + B_TIME * transit_time + B_COST_LOW * transit_fare * hh_share_inc_under_100k + B_COST_HI * transit_fare * hh_share_inc_over_100k 
V_walk    = ASC_WALK    + B_TIME * walk_time

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit, 3: V_walk}
avail = {1: 1, 2: Variable('transit_avail'), 3: Variable('walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, CHOICE)
the_biogeme = bio.BIOGEME(db, logprob)
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))



Results for model mnl_mode_choice
Nbr of parameters:		5
Sample size:			5769
Excluded data:			0
Null log likelihood:		-5835.602
Final log likelihood:		-2501.275
Likelihood ratio test (null):		6668.655
Rho square (null):			0.571
Rho bar square (null):			0.571
Akaike Information Criterion:	5012.55
Bayesian Information Criterion:	5045.851

                Value  Rob. Std err  Rob. t-test  Rob. p-value
ASC_TRANSIT  2.040729      0.121777    16.757847  0.000000e+00
ASC_WALK     2.928676      0.154142    18.999862  0.000000e+00
B_COST_HI   -0.430531      0.045695    -9.421795  0.000000e+00
B_COST_LOW  -0.266744      0.038923    -6.853150  7.224221e-12
B_TIME      -0.091805      0.006822   -13.458195  0.000000e+00

Value of Time for HH <$100k: 20.65
Value of Time for HH $100k+: 12.79


# What happens if I estimate a binary choice instead? 

Here it is just the choice of TNC vs the faster of transit or walk. This gives a much higher value of time. 

In [ ]:
# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT_WALK = Beta('ASC_TRANSIT_WALK', 0, None, None, 0)
B_TIME           = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST           = Beta('B_COST',      0, None, None, 0)   # generic, shared across modes

# --- variables ---
tnc_time             = Variable('tnc_time')
transit_or_walk_time = Variable('transit_or_walk_time')
tnc_fare             = Variable('tnc_fare')
transit_or_walk_fare = Variable('transit_or_walk_fare')
BINARY_CHOICE        = Variable('BINARY_CHOICE')

# --- utility equations ---
V_tnc     =                         B_TIME * tnc_time             + B_COST * tnc_fare
V_transit_walk = ASC_TRANSIT_WALK + B_TIME * transit_or_walk_time + B_COST * transit_or_walk_fare

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit_walk}
avail = {1: 1, 2: Variable('transit_or_walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, BINARY_CHOICE)
the_biogeme = bio.BIOGEME(db, logprob)
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST']['Value']
print("\nValue of Time: " + str(round(vot, 2)))


# Binary choice with segmented HH income


In [ ]:
# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT_WALK = Beta('ASC_TRANSIT_WALK', 0, None, None, 0)
B_TIME      = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW  = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI   = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers
B_COST_MISSING = Beta('B_COST_MISSING', 0, None, None, 0)   # cost coefficient for travelers with income missing or not reported

# --- variables ---
tnc_time             = Variable('tnc_time')
transit_or_walk_time = Variable('transit_or_walk_time')
tnc_fare             = Variable('tnc_fare')
transit_or_walk_fare = Variable('transit_or_walk_fare')

inc_under_100k = Variable('inc_under_100k')
inc_over_100k  = Variable('inc_over_100k')
inc_missing    = Variable('inc_missing')

BINARY_CHOICE        = Variable('BINARY_CHOICE')

# --- utility equations ---
V_tnc     = (             B_TIME * tnc_time     
                        + B_COST_LOW * tnc_fare * inc_under_100k 
                        + B_COST_HI * tnc_fare * inc_over_100k 
                        + B_COST_MISSING * tnc_fare * inc_missing)

V_transit_walk = (ASC_TRANSIT + B_TIME * transit_or_walk_time 
                         + B_COST_LOW * transit_or_walk_fare * inc_under_100k 
                         + B_COST_HI * transit_or_walk_fare * inc_over_100k 
                         + B_COST_MISSING * transit_or_walk_fare * inc_missing)

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit_walk}
avail = {1: 1, 2: Variable('transit_or_walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, BINARY_CHOICE)
the_biogeme = bio.BIOGEME(db, logprob)
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_MISSING']['Value']
print("Value of Time for HH with missing income: " + str(round(vot, 2)))


# Binary choice with income segmented based on zonal HH income distribution



In [ ]:
# --- coefficients ---
# Name, starting value, lower bound, upper bound, status (0=estimate, 1=fixed)
ASC_TRANSIT_WALK = Beta('ASC_TRANSIT_WALK', 0, None, None, 0)
ASC_WALK         = Beta('ASC_WALK',    0, None, None, 0)   
B_TIME           = Beta('B_TIME',      0, None, None, 0)   # generic, shared across modes
B_COST_LOW       = Beta('B_COST_LOW',  0, None, None, 0)   # cost coefficient for lower income travelers
B_COST_HI        = Beta('B_COST_HI',   0, None, None, 0)   # cost coefficient for higher income travelers

# --- variables ---
tnc_time             = Variable('tnc_time')
transit_or_walk_time = Variable('transit_or_walk_time')
tnc_fare             = Variable('tnc_fare')
transit_or_walk_fare = Variable('transit_or_walk_fare')

hh_share_inc_under_100k = Variable('hh_share_inc_under_100k')
hh_share_inc_over_100k  = Variable('hh_share_inc_over_100k')

BINARY_CHOICE        = Variable('BINARY_CHOICE')

# --- utility equations ---
V_tnc     =               B_TIME * tnc_time     + B_COST_LOW * tnc_fare * hh_share_inc_under_100k + B_COST_HI * tnc_fare * hh_share_inc_over_100k 
V_transit_walk = (ASC_TRANSIT + B_TIME * transit_or_walk_time 
                              + B_COST_LOW * transit_or_walk_fare * hh_share_inc_under_100k 
                              + B_COST_HI * transit_or_walk_fare * hh_share_inc_over_100k )

# specify which equations align with which alternatives, and the availability of each alternative
V  = {1: V_tnc, 2: V_transit_walk}
avail = {1: 1, 2: Variable('transit_or_walk_avail')}          # all alternatives available for every trip

# --- estimate ---
logprob = models.loglogit(V, avail, BINARY_CHOICE)
the_biogeme = bio.BIOGEME(db, logprob)
the_biogeme.modelName = 'mnl_mode_choice'
the_biogeme.calculate_null_loglikelihood(avail=avail)
results = the_biogeme.estimate()

# --- print results ---
print(results.short_summary())
print(results.get_estimated_parameters())

# --- print value of time ---
params = results.get_estimated_parameters()
vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_LOW']['Value']
print("\nValue of Time for HH <$100k: " + str(round(vot, 2)))

vot = 60 * params.loc['B_TIME']['Value'] / params.loc['B_COST_HI']['Value']
print("Value of Time for HH $100k+: " + str(round(vot, 2)))

